In [41]:
import numpy as np
from pathlib import Path
import pandas as pd
import shutil
import re
import sleap_io as sio

# ── Configuration ─────────────────────────────────────────────────────
SLP_FILE = Path(r"C:\Academic\Research\PigVLM\sleap_label\ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s\labels.v002.slp")  # ← Update this
DLC_PROJECT = Path(r"C:\Academic\Research\PigVLM\PigFarm_Sow-jiale-2026-02-08")  # ← Update this
SCORER = "jiale"  # Must match config.yaml scorer field

# If SLEAP's image folder name doesn't match the DLC labeled-data subdir,
# provide an explicit mapping: {"sleap_folder_name": "dlc_video_name"}
# Leave empty to auto-detect from folder names.
FOLDER_NAME_MAP = {}

# ── Load SLEAP labels ────────────────────────────────────────────────
labels = sio.load_slp(str(SLP_FILE))
skeleton = labels.skeletons[0]
bodyparts = [node.name for node in skeleton.nodes]

print(f"Skeleton bodyparts: {bodyparts}")
print(f"Labeled frames: {len(labels.labeled_frames)}")
print(f"Videos (image folders): {len(labels.videos)}")

for v in labels.videos:
    print(f"  {v.filename}")

def parse_frame_index(filename: str) -> int:
    """Extract the frame index number from various naming patterns.

    Handles:
      - "frame_000050.png" → 50
      - "img0050.png"      → 50
      - "img050.png"       → 50
      - "000050.png"       → 50
    """
    match = re.search(r"(\d+)", Path(filename).stem)
    if match:
        return int(match.group(1))
    raise ValueError(f"Cannot parse frame index from: {filename}")

def get_dlc_video_name(sleap_video_path: Path) -> str:
    """Map SLEAP video (image folder) path to DLC labeled-data subdirectory name."""
    folder_name = sleap_video_path.name if sleap_video_path.is_dir() else sleap_video_path.stem
    return FOLDER_NAME_MAP.get(folder_name, folder_name)

Skeleton bodyparts: ['left_ear', 'right_ear', 'torso', 'hip']
Labeled frames: 20
Videos (image folders): 20
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s/img022.png')]
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s/img054.png')]
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s/img069.png')]
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s/img093.png')]
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s/img150.png')]
  [WindowsPath('C:/Academic/Research/PigVLM/PigFarm_Sow-jiale-2026-02-08/labeled-data/ch07_Crate08_202

In [42]:
# ── Resolve source images per labeled frame ──────────────────────────
# SLEAP may store videos as:
#   (a) an image folder  → video.filename is a directory
#   (b) individual files → video.filename is a single image path
# We handle both by resolving each labeled frame to its source image.

from collections import defaultdict

# Group labeled frames by their DLC video subdirectory
# (determined by the parent folder of the source image)
groups = defaultdict(list)  # video_name → list of (src_image_path, lf)

for lf in labels.labeled_frames:
    if len(lf.instances) == 0:
        print(f"WARNING: frame with no instances, skipping")
        continue

    src_path = Path(lf.video.filename[0])

    if src_path.is_dir():
        # Case (a): video is an image folder — map frame_idx to sorted file list
        src_images = sorted(
            list(src_path.glob("*.png")) + list(src_path.glob("*.jpg"))
        )
        if lf.frame_idx >= len(src_images):
            print(f"WARNING: frame_idx {lf.frame_idx} out of range in {src_path.name}, skipping")
            continue
        src_image = src_images[lf.frame_idx]
        video_name = src_path.name
    else:
        # Case (b): video is a single image file
        src_image = src_path
        video_name = src_path.parent.name

    video_name = FOLDER_NAME_MAP.get(video_name, video_name)
    groups[video_name].append((src_image, lf))

print(f"\nGrouped into {len(groups)} video folder(s):")
for name, items in groups.items():
    print(f"  {name}: {len(items)} labeled frames")

# ── Process each group ────────────────────────────────────────────────
total_frames = 0

for video_name, frame_list in groups.items():
    labeled_data_dir = DLC_PROJECT / "labeled-data" / video_name
    labeled_data_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'─'*60}")
    print(f"Video: {video_name}")
    print(f"Target: {labeled_data_dir}")

    rows = {}
    for src_image, lf in frame_list:
        # Preserve original filename if it's already DLC format (img*.png),
        # otherwise rename from other patterns (frame_*.png, etc.)
        if src_image.name.startswith("img"):
            dlc_img_name = src_image.name
        else:
            orig_idx = parse_frame_index(src_image.name)
            dlc_img_name = f"img{orig_idx:04d}.png"
        dlc_img_path = labeled_data_dir / dlc_img_name

        # Copy image to DLC directory (skip if already present)
        if not dlc_img_path.exists() and src_image.exists():
            shutil.copy2(src_image, dlc_img_path)

        # DLC relative path (used as CSV row index)
        rel_path = f"labeled-data/{video_name}/{dlc_img_name}"

        # Extract coordinates from the first (only) instance
        inst = lf.instances[0]
        coords = inst.numpy()  # shape: (n_nodes, 2)

        row_data = {}
        for i, bp in enumerate(bodyparts):
            x, y = coords[i]
            row_data[(SCORER, bp, "x")] = x if not np.isnan(x) else ""
            row_data[(SCORER, bp, "y")] = y if not np.isnan(y) else ""

        rows[rel_path] = row_data

    if not rows:
        print("  No labeled frames to export, skipping.")
        continue

    # Build DataFrame with DLC's expected MultiIndex columns
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.columns = pd.MultiIndex.from_tuples(
        df.columns, names=["scorer", "bodyparts", "coords"]
    )
    df = df.sort_index()

    # Save CSV
    csv_path = labeled_data_dir / f"CollectedData_{SCORER}.csv"
    df.to_csv(csv_path)

    total_frames += len(df)
    print(f"  Saved: {csv_path}")
    print(f"  Frames exported: {len(df)}")

# ── Validate: every CSV path must have a matching image on disk ───────
print(f"\n{'='*60}")
print("Validating image paths...")

missing = []
for video_name, frame_list in groups.items():
    labeled_data_dir = DLC_PROJECT / "labeled-data" / video_name
    csv_path = labeled_data_dir / f"CollectedData_{SCORER}.csv"
    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path, header=[0, 1, 2], index_col=0)
    for rel_path in df.index:
        abs_path = DLC_PROJECT / rel_path
        if not abs_path.exists():
            missing.append(rel_path)

if missing:
    print(f"\nERROR: {len(missing)} image(s) referenced in CSV but NOT found on disk:")
    for m in missing:
        print(f"  MISSING: {m}")
    print("\nThe CSV references files that don't exist. DLC will fail during training.")
    print("Check that image filenames in labeled-data/ match the CSV entries.")
    raise SystemExit(1)
else:
    print(f"All {total_frames} image paths verified — every CSV entry has a matching file.")

# ── Summary ───────────────────────────────────────────────────────────
print(f"\nConversion complete! {total_frames} total frames exported.")
print(f"\nNext step — finalize with DLC (on server in dlc conda env):")
print(f"  import deeplabcut")
print(f"  deeplabcut.convertcsv2h5('{DLC_PROJECT}/config.yaml', scorer='{SCORER}')")


Grouped into 1 video folder(s):
  ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s: 20 labeled frames

────────────────────────────────────────────────────────────
Video: ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s
Target: C:\Academic\Research\PigVLM\PigFarm_Sow-jiale-2026-02-08\labeled-data\ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s
  Saved: C:\Academic\Research\PigVLM\PigFarm_Sow-jiale-2026-02-08\labeled-data\ch07_Crate08_20220428080001_20220428100000_clip_00h25m00s\CollectedData_jiale.csv
  Frames exported: 20

Validating image paths...
All 20 image paths verified — every CSV entry has a matching file.

Conversion complete! 20 total frames exported.

Next step — finalize with DLC (on server in dlc conda env):
  import deeplabcut
  deeplabcut.convertcsv2h5('C:\Academic\Research\PigVLM\PigFarm_Sow-jiale-2026-02-08/config.yaml', scorer='jiale')
